In [ ]:
import os
import zipfile
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')
ZIP_PATH = '/content/drive/My Drive/dataset_top10.zip'
EXTRACT_DIR = '/content/dataset_top10'

if not os.path.exists(EXTRACT_DIR):
    print(f"Начинаем распаковку {ZIP_PATH} в {EXTRACT_DIR}...")
    with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
        zip_ref.extractall('/content/')
    print("Распаковка успешно завершена!\n")
else:
    print("Датасет уже распакован.\n")

# Проверка структуры
TRAIN_DIR = Path(EXTRACT_DIR) / "train"
TEST_DIR = Path(EXTRACT_DIR) / "test"

def count_images(directory):
    if not directory.exists():
        return 0
    return sum(1 for p in directory.rglob("*") if p.suffix.lower() in {".jpg", ".jpeg", ".png"})

print("=" * 40)
print("СТРУКТУРА ДАТАСЕТА:")
print(f"Train/Val директория: {count_images(TRAIN_DIR):,} изображений")
print(f"Test директория:      {count_images(TEST_DIR):,} изображений")
print("=" * 40)

class_names = sorted([d.name for d in TRAIN_DIR.iterdir() if d.is_dir()])
print(f"\nКлассы ({len(class_names)}):")
for i, name in enumerate(class_names, 1):
    print(f"{i}. {name}")

Mounted at /content/drive
Начинаем распаковку /content/drive/My Drive/dataset_top10.zip в /content/dataset_top10...
Распаковка успешно завершена!

СТРУКТУРА ДАТАСЕТА:
Train/Val директория: 48,447 изображений
Test директория:      12,118 изображений

Классы (10):
1. Abstract_Expressionism
2. Art_Nouveau_Modern
3. Baroque
4. Expressionism
5. Impressionism
6. Northern_Renaissance
7. Post_Impressionism
8. Realism
9. Romanticism
10. Symbolism


## 1. Настройки и аугментация

In [ ]:
import tensorflow as tf
from pathlib import Path
DATASET_DIR  = Path("/content/dataset_top10")
TRAIN_DIR    = DATASET_DIR / "train"
TEST_DIR     = DATASET_DIR / "test"

BATCH_SIZE   = 32
RANDOM_SEED  = 42
AUTOTUNE     = tf.data.AUTOTUNE
VAL_SPLIT    = 0.15          # 15 % от train уходит в val

# Слой аугментации (применяется ТОЛЬКО к train)
augmentation_layer = tf.keras.Sequential([
    tf.keras.layers.RandomRotation(0.02),
    tf.keras.layers.RandomZoom(0.1),
    tf.keras.layers.RandomTranslation(height_factor=0.1, width_factor=0.1),
], name="augmentation")

print("Глобальные настройки и слой аугментации загружены.")

Глобальные настройки и слой аугментации загружены.


**Основная функция**

In [ ]:
def build_datasets(preprocess_fn, img_size=(224, 224)):
    """
    Собирает tf.data пайплайн для train / val / test.
    """
    # train split
    train_ds = tf.keras.utils.image_dataset_from_directory(
        TRAIN_DIR,
        image_size=img_size,
        batch_size=BATCH_SIZE,
        shuffle=True,
        seed=RANDOM_SEED,
        label_mode="categorical",
        validation_split=VAL_SPLIT,
        subset="training",
    )

    # val split
    val_ds = tf.keras.utils.image_dataset_from_directory(
        TRAIN_DIR,
        image_size=img_size,
        batch_size=BATCH_SIZE,
        shuffle=False,
        seed=RANDOM_SEED,
        label_mode="categorical",
        validation_split=VAL_SPLIT,
        subset="validation",
    )

    # test
    test_ds = tf.keras.utils.image_dataset_from_directory(
        TEST_DIR,
        image_size=img_size,
        batch_size=BATCH_SIZE,
        shuffle=False,
        label_mode="categorical",
    )

    class_names = train_ds.class_names

    # аугментация + предобработка
    def train_preprocess(images, labels):
        images = augmentation_layer(images, training=True)
        images = preprocess_fn(images)
        return images, labels

    def eval_preprocess(images, labels):
        images = preprocess_fn(images)
        return images, labels

    train_ds = (
        train_ds
        .map(train_preprocess, num_parallel_calls=AUTOTUNE)
        .prefetch(AUTOTUNE)
    )
    val_ds = (
        val_ds
        .cache()
        .map(eval_preprocess, num_parallel_calls=AUTOTUNE)
        .prefetch(AUTOTUNE)
    )
    test_ds = (
        test_ds
        .cache()
        .map(eval_preprocess, num_parallel_calls=AUTOTUNE)
        .prefetch(AUTOTUNE)
    )

    # статистика
    print("=" * 58)
    print(f"{'Split':<8} | {'Батчей/эпоха':>14}")
    print("-" * 58)
    for name, ds in [("Train", train_ds), ("Val", val_ds), ("Test", test_ds)]:
        batches = ds.cardinality().numpy()
        print(f"{name:<8} | {batches:>14,}")
    print("=" * 58)
    print(f"\nIMG_SIZE   = {img_size}")
    print(f"BATCH_SIZE = {BATCH_SIZE}")
    print(f"Классы ({len(class_names)}): {class_names}")

    return train_ds, val_ds, test_ds, class_names

print("Функция build_datasets готова к работе.")

Функция build_datasets готова к работе.


**Пример вызова: EfficientNetB0**

In [ ]:
from tensorflow.keras.applications.efficientnet import preprocess_input as effnet_preprocess

print("Сборка пайплайна для EfficientNetB0...")
train_ds, val_ds, test_ds, class_names = build_datasets(
    preprocess_fn=effnet_preprocess,
    img_size=(224, 224),
)

# Быстрая проверка: берём один батч и смотрим shape + диапазон
for images, labels in train_ds.take(1):
    print(f"\nПроверка первого батча (Train):")
    print(f"  images.shape = {images.shape}")
    print(f"  labels.shape = {labels.shape}")
    print(f"  pixel range  = [{images.numpy().min():.1f}, {images.numpy().max():.1f}]")

Сборка пайплайна для EfficientNetB0...
Found 48447 files belonging to 10 classes.
Using 41180 files for training.
Found 48447 files belonging to 10 classes.
Using 7267 files for validation.
Found 12118 files belonging to 10 classes.
Split    |   Батчей/эпоха
----------------------------------------------------------
Train    |          1,287
Val      |            228
Test     |            379

IMG_SIZE   = (224, 224)
BATCH_SIZE = 32
Классы (10): ['Abstract_Expressionism', 'Art_Nouveau_Modern', 'Baroque', 'Expressionism', 'Impressionism', 'Northern_Renaissance', 'Post_Impressionism', 'Realism', 'Romanticism', 'Symbolism']

Проверка первого батча (Train):
  images.shape = (32, 224, 224, 3)
  labels.shape = (32, 10)
  pixel range  = [0.1, 254.8]


# 2. Универсальная функция построения моделей

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# backbone-конфигурации
BACKBONE_CONFIG = {
    "ResNet50": {
        "model_fn":       tf.keras.applications.ResNet50,
        "preprocess_fn":  tf.keras.applications.resnet50.preprocess_input,
        "default_size":   (224, 224),
    },
    "Xception": {
        "model_fn":       tf.keras.applications.Xception,
        "preprocess_fn":  tf.keras.applications.xception.preprocess_input,
        "default_size":   (299, 299),
    },
    "EfficientNetB0": {
        "model_fn":       tf.keras.applications.EfficientNetB0,
        "preprocess_fn":  tf.keras.applications.efficientnet.preprocess_input,
        "default_size":   (224, 224),
    },
}

print("Конфигурация архитектур успешно загружена.")

Конфигурация архитектур успешно загружена.


In [ ]:
def build_model(backbone_name, num_classes=10, img_size=None):
    """
    Собирает модель: замороженный backbone + классификационная голова.
    """
    if backbone_name not in BACKBONE_CONFIG:
        raise ValueError(
            f"Неизвестный backbone: '{backbone_name}'. "
            f"Допустимые: {list(BACKBONE_CONFIG.keys())}"
        )

    cfg = BACKBONE_CONFIG[backbone_name]
    recommended = cfg["default_size"]

    # Проверка
    if img_size is None:
        img_size = recommended
        print(f"[INFO] img_size не указан → используем рекомендованный "
              f"для {backbone_name}: {img_size}")
    elif img_size != recommended:
        print(f" ВНИМАНИЕ: для {backbone_name} рекомендуется "
              f"img_size={recommended}, а передан {img_size}. "
              f"Качество может пострадать!")

    input_shape = img_size + (3,)

    # 1. Backbone
    backbone = cfg["model_fn"](
        include_top=False,
        weights="imagenet",
        input_shape=input_shape,
    )
    backbone.trainable = False         # замораживаем все слои

    # 2. Классификатор
    inputs  = keras.Input(shape=input_shape, name="input_image")
    x = backbone(inputs, training=False)   # training=False → BN в inference-режиме
    x = layers.GlobalAveragePooling2D(name="gap")(x)
    x = layers.BatchNormalization(name="bn_1")(x)
    x = layers.Dropout(0.3, name="drop_1")(x)
    x = layers.Dense(256, activation="relu", name="fc_256")(x)
    x = layers.BatchNormalization(name="bn_2")(x)
    x = layers.Dropout(0.3, name="drop_2")(x)
    outputs = layers.Dense(num_classes, activation="softmax", name="predictions")(x)

    model = keras.Model(inputs, outputs, name=f"{backbone_name}_classifier")

    # 3. Компиляция
    # Метрика создается ЗДЕСЬ, индивидуально для каждой модели!
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=1e-3),
        loss="categorical_crossentropy",
        metrics=[
            "accuracy",
            tf.keras.metrics.TopKCategoricalAccuracy(k=3, name="top3_accuracy"),
        ],
    )

    # 4. Сводка
    total   = model.count_params()
    trainable = sum(tf.keras.backend.count_params(w)
                    for w in model.trainable_weights)
    frozen  = total - trainable

    print(f"\n{'=' * 55}")
    print(f"  {backbone_name}  |  input {img_size}  |  {num_classes} классов")
    print(f"  Всего параметров:      {total:>12,}")
    print(f"  Обучаемых (голова):    {trainable:>12,}")
    print(f"  Замороженных (backbone): {frozen:>12,}")
    print(f"{'=' * 55}\n")

    return model, cfg["preprocess_fn"], img_size

print("Функция build_model готова к работе.")

Функция build_model готова к работе.


In [ ]:
# summary для всех трёх моделей

for name in ["ResNet50", "Xception", "EfficientNetB0"]:
    model, preprocess_fn, size = build_model(name)
    model.summary(show_trainable=True, expand_nested=False)
    print("\n" + "─" * 70 + "\n")
    del model
    tf.keras.backend.clear_session()

[INFO] img_size не указан → используем рекомендованный для ResNet50: (224, 224)
94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step

  ResNet50  |  input (224, 224)  |  10 классов
  Всего параметров:        24,124,042
  Обучаемых (голова):         531,722
  Замороженных (backbone):   23,592,320



Model: "ResNet50_classifier"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━┓
┃ Layer (type)                ┃ Output Shape          ┃    Param # ┃ Trai… ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━┩
│ input_image (InputLayer)    │ (None, 224, 224, 3)   │          0 │   -   │
├─────────────────────────────┼───────────────────────┼────────────┼───────┤
│ resnet50 (Functional)       │ (None, 7, 7, 2048)    │ 23,587,712 │   N   │
├─────────────────────────────┼───────────────────────┼────────────┼───────┤
│ gap                         │ (None, 2048)          │          0 │   -   │
│ (GlobalAveragePooling2D)    │                       │            │       │
├─────────────────────────────┼───────────────────────┼────────────┼───────┤
│ bn_1 (BatchNormalization)   │ (None, 2048)          │      8,192 │   Y   │
├─────────────────────────────┼───────────────────────┼────────────┼───────┤
│ drop_1 (Dropout)            │ (None, 2048)          │          0 │   -   │
├─────────────────────────────┼───────────────────────┼────────────┼───────┤
│ fc_256 (Dense)              │ (None, 256)           │    524,544 │   Y   │
├─────────────────────────────┼───────────────────────┼────────────┼───────┤
│ bn_2 (BatchNormalization)   │ (None, 256)           │      1,024 │   Y   │
├─────────────────────────────┼───────────────────────┼────────────┼───────┤
│ drop_2 (Dropout)            │ (None, 256)           │          0 │   -   │
├─────────────────────────────┼───────────────────────┼────────────┼───────┤
│ predictions (Dense)         │ (None, 10)            │      2,570 │   Y   │
└─────────────────────────────┴───────────────────────┴────────────┴───────┘

 Total params: 24,124,042 (92.03 MB)

 Trainable params: 531,722 (2.03 MB)

...

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━┓
┃ Layer (type)                ┃ Output Shape          ┃    Param # ┃ Trai… ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━┩
│ input_image (InputLayer)    │ (None, 224, 224, 3)   │          0 │   -   │
├─────────────────────────────┼───────────────────────┼────────────┼───────┤
│ efficientnetb0 (Functional) │ (None, 7, 7, 1280)    │  4,049,571 │   N   │
├─────────────────────────────┼───────────────────────┼────────────┼───────┤
│ gap                         │ (None, 1280)          │          0 │   -   │
│ (GlobalAveragePooling2D)    │                       │            │       │
├─────────────────────────────┼───────────────────────┼────────────┼───────┤
│ bn_1 (BatchNormalization)   │ (None, 1280)          │      5,120 │   Y   │
├─────────────────────────────┼───────────────────────┼────────────┼───────┤
│ drop_1 (Dropout)            │ (None, 1280)          │          0 │   -   │
├─────────────────────────────┼───────────────────────┼────────────┼───────┤
│ fc_256 (Dense)              │ (None, 256)           │    327,936 │   Y   │
├─────────────────────────────┼───────────────────────┼────────────┼───────┤
│ bn_2 (BatchNormalization)   │ (None, 256)           │      1,024 │   Y   │
├─────────────────────────────┼───────────────────────┼────────────┼───────┤
│ drop_2 (Dropout)            │ (None, 256)           │          0 │   -   │
├─────────────────────────────┼───────────────────────┼────────────┼───────┤
│ predictions (Dense)         │ (None, 10)            │      2,570 │   Y   │
└─────────────────────────────┴───────────────────────┴────────────┴───────┘

 Total params: 4,386,221 (16.73 MB)

 Trainable params: 333,578 (1.27 MB)

 Non-trainable params: 4,052,643 (15.46 MB)


──────────────────────────────────────────────────────────────────────



# 3. Функция обучения с колбэками

In [ ]:
import os
import tensorflow as tf
from tensorflow import keras
from pathlib import Path

def train_model(model, backbone_name, train_ds, val_ds, save_dir,
                phase1_epochs=7, phase2_epochs=15,
                phase1_lr=1e-3, phase2_lr=1e-5,
                unfreeze_fraction=0.3):

    save_dir = Path(save_dir)
    save_dir.mkdir(parents=True, exist_ok=True)

    ckpt_path = save_dir / f"{backbone_name}_best.keras"
    csv_path  = save_dir / f"{backbone_name}_history.csv"

    # Находим backbone-слой внутри модели
    backbone = None
    for layer in model.layers:
        if isinstance(layer, tf.keras.Model):
            backbone = layer
            break
    if backbone is None:
        raise RuntimeError("Не удалось найти backbone внутри модели.")

    print(f"Backbone: {backbone.name}  |  слоёв: {len(backbone.layers)}")


    #  ФАЗА 1: обучение головы

    print("\n" + "=" * 60)
    print(f"  ФАЗА 1  |  {backbone_name}  |  голова  |  lr={phase1_lr}")
    print("=" * 60)

    backbone.trainable = False

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=phase1_lr),
        loss="categorical_crossentropy",
        metrics=[
            "accuracy",
            tf.keras.metrics.TopKCategoricalAccuracy(k=3, name="top3_accuracy"),
        ],
    )

    callbacks_p1 = [
        keras.callbacks.ModelCheckpoint(
            str(ckpt_path), monitor="val_accuracy",
            save_best_only=True, mode="max", verbose=1
        ),
        keras.callbacks.EarlyStopping(
            monitor="val_loss", patience=3,
            restore_best_weights=True, verbose=1
        ),
        keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss", factor=0.5,
            patience=2, min_lr=1e-7, verbose=1
        ),
        keras.callbacks.CSVLogger(str(csv_path), append=False),
    ]

    hist1 = model.fit(
        train_ds, validation_data=val_ds,
        epochs=phase1_epochs, callbacks=callbacks_p1, verbose=1
    )

    phase1_best_val = max(hist1.history["val_accuracy"])
    print(f"\n✓ Фаза 1 завершена.  Лучшая val_accuracy = {phase1_best_val:.4f}")


    #  ФАЗА 2: fine-tuning

    print("\n" + "=" * 60)
    print(f"  ФАЗА 2  |  {backbone_name}  |  fine-tuning  |  lr={phase2_lr}")
    print("=" * 60)

    backbone.trainable = True
    total_layers  = len(backbone.layers)
    freeze_until  = int(total_layers * (1 - unfreeze_fraction))

    for layer in backbone.layers[:freeze_until]:
        layer.trainable = False

    trainable_count = sum(1 for l in backbone.layers if l.trainable)
    print(f"  Всего слоёв backbone:    {total_layers}")
    print(f"  Заморожено (первые):     {freeze_until}")
    print(f"  Разморожено (последние): {trainable_count}")

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=phase2_lr),
        loss="categorical_crossentropy",
        metrics=[
            "accuracy",
            tf.keras.metrics.TopKCategoricalAccuracy(k=3, name="top3_accuracy"),
        ],
    )

    callbacks_p2 = [
        keras.callbacks.ModelCheckpoint(
            str(ckpt_path), monitor="val_accuracy",
            save_best_only=True, mode="max", verbose=1
        ),
        keras.callbacks.EarlyStopping(
            monitor="val_loss", patience=3,
            restore_best_weights=True, verbose=1
        ),
        keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss", factor=0.5,
            patience=2, min_lr=1e-7, verbose=1
        ),
        keras.callbacks.CSVLogger(str(csv_path), append=True),
    ]

    hist2 = model.fit(
        train_ds, validation_data=val_ds,
        epochs=phase2_epochs, callbacks=callbacks_p2, verbose=1
    )

    phase2_best_val = max(hist2.history["val_accuracy"])
    print(f"\n✓ Фаза 2 завершена.  Лучшая val_accuracy = {phase2_best_val:.4f}")

    #  Склейка истории
    history = {}
    for key in hist1.history:
        history[key] = hist1.history[key] + hist2.history[key]

    history["phase1_epochs"] = len(hist1.history["loss"])
    overall_best = max(history["val_accuracy"])
    best_epoch   = history["val_accuracy"].index(overall_best) + 1

    print(f"\n{'─' * 60}")
    print(f"  ИТОГ  {backbone_name}")
    print(f"  Лучшая val_accuracy = {overall_best:.4f}  (эпоха {best_epoch})")
    print(f"  Чекпоинт: {ckpt_path}")
    print(f"  История:  {csv_path}")
    print(f"{'─' * 60}\n")

    return model, history

print("Полная функция train_model успешно загружена в память!")

Полная функция train_model успешно загружена в память!


In [ ]:
from pathlib import Path

SAVE_DIR = Path("/content/drive/MyDrive/art_classification/checkpoints")
print(f"Папка существует: {SAVE_DIR.exists()}")

if SAVE_DIR.exists():
    for f in sorted(SAVE_DIR.iterdir()):
        size = f.stat().st_size / (1024*1024)
        print(f"  {f.name:<40} {size:.1f} МБ")
else:
    print("Drive не примонтирован! Запусти ячейку 0 (drive.mount)")

Папка существует: True
  EfficientNetB0_best.keras                43.6 МБ
  EfficientNetB0_history.csv               0.0 МБ
  ResNet50_best.keras                      228.1 МБ
  ResNet50_history.csv                     0.0 МБ
  Xception_best.keras                      166.7 МБ
  Xception_history.csv                     0.0 МБ


In [ ]:
# УНИВЕРСАЛЬНОЕ ДООБУЧЕНИЕ (Фаза 2)
import gc
import tensorflow as tf
from tensorflow import keras
from pathlib import Path

SAVE_DIR = "/content/drive/MyDrive/art_classification/checkpoints"

PHASE2_EPOCHS = 15

models_to_finish = {
    "EfficientNetB0": {
        "preprocess": tf.keras.applications.efficientnet.preprocess_input,
        "img_size": (224, 224),
    },
    "ResNet50": {
        "preprocess": tf.keras.applications.resnet50.preprocess_input,
        "img_size": (224, 224),
    },
    "Xception": {
        "preprocess": tf.keras.applications.xception.preprocess_input,
        "img_size": (299, 299),
    },
}

def is_fully_trained(name):
    return (Path(SAVE_DIR) / f"{name}_DONE.txt").exists()

def mark_done(name):
    (Path(SAVE_DIR) / f"{name}_DONE.txt").write_text(f"{name} training completed")

for name, cfg in models_to_finish.items():
    if is_fully_trained(name):
        print(f"⏭️ [ПРОПУСК] {name} — уже достигла предела (EarlyStopping сработал ранее)")
        continue

    ckpt_path = Path(SAVE_DIR) / f"{name}_best.keras"
    csv_path  = Path(SAVE_DIR) / f"{name}_history.csv"

    if not ckpt_path.exists():
        print(f" Чекпоинт {name} не найден. Пропускаем.")
        continue

    print(f"\n{'=' * 60}")
    print(f"  ДООБУЧЕНИЕ ФАЗЫ 2  |  {name}")
    print(f"{'=' * 60}")

    # 1. Загружаем модель (сохраняя состояние оптимизатора!)
    model = keras.models.load_model(str(ckpt_path))

    # 2. Умная проверка Фазы 2
    # Ищем backbone и проверяем, есть ли в нем размороженные слои
    backbone = next((layer for layer in model.layers if isinstance(layer, tf.keras.Model)), None)

    is_phase_2 = any(l.trainable for l in backbone.layers) if backbone else False

    if not is_phase_2 and backbone:
        print("  -> Модель была в Фазе 1. Размораживаем 30% слоев и компилируем...")
        backbone.trainable = True
        freeze_until = int(len(backbone.layers) * 0.7)
        for layer in backbone.layers[:freeze_until]:
            layer.trainable = False

        model.compile(
            optimizer=keras.optimizers.Adam(learning_rate=1e-5),
            loss="categorical_crossentropy",
            metrics=["accuracy", tf.keras.metrics.TopKCategoricalAccuracy(k=3, name="top3_accuracy")]
        )
    else:
        print("  -> Модель УЖЕ в Фазе 2. Продолжаем обучение без сброса оптимизатора!")
        # Мы НЕ вызываем model.compile(), чтобы сохранить инерцию Adam

    # 3. Подготовка датасетов (с правильным размером картинки для текущей сети)
    train_ds = tf.keras.utils.image_dataset_from_directory(
        TRAIN_DIR, image_size=cfg["img_size"], batch_size=32,
        shuffle=True, seed=42, label_mode="categorical",
        validation_split=0.15, subset="training",
    )
    val_ds = tf.keras.utils.image_dataset_from_directory(
        TRAIN_DIR, image_size=cfg["img_size"], batch_size=32,
        shuffle=False, seed=42, label_mode="categorical",
        validation_split=0.15, subset="validation",
    )

    aug = tf.keras.Sequential([
        tf.keras.layers.RandomRotation(0.02),
        tf.keras.layers.RandomZoom(0.1),
        tf.keras.layers.RandomTranslation(0.1, 0.1),
    ])

    pfn = cfg["preprocess"]
    train_ds = train_ds.map(lambda x, y: (pfn(aug(x, training=True)), y), num_parallel_calls=tf.data.AUTOTUNE).prefetch(tf.data.AUTOTUNE)
    val_ds = val_ds.map(lambda x, y: (pfn(x), y), num_parallel_calls=tf.data.AUTOTUNE).prefetch(tf.data.AUTOTUNE)

    # 4. Запуск!
    history = model.fit(
        train_ds, validation_data=val_ds,
        epochs=PHASE2_EPOCHS,
        callbacks=[
            keras.callbacks.ModelCheckpoint(str(ckpt_path), monitor="val_accuracy", save_best_only=True, mode="max", verbose=1),
            keras.callbacks.EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True, verbose=1),
            keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2, min_lr=1e-7, verbose=1),
            keras.callbacks.CSVLogger(str(csv_path), append=True),
        ], verbose=1,
    )

    # 5. Помечаем как полностью готовую
    mark_done(name)
    print(f"\n✓ Обучение {name} официально завершено алгоритмом EarlyStopping!")

    # Очистка памяти перед следующей моделью
    del model, train_ds, val_ds
    tf.keras.backend.clear_session()
    gc.collect()

print("\n🎉 ВСЕ МОДЕЛИ ДОСТИГЛИ СВОЕГО АБСОЛЮТНОГО ПРЕДЕЛА!")

⏭️ [ПРОПУСК] EfficientNetB0 — уже достигла предела (EarlyStopping сработал ранее)

  ДООБУЧЕНИЕ ФАЗЫ 2  |  ResNet50
  -> Модель УЖЕ в Фазе 2. Продолжаем обучение без сброса оптимизатора!
Found 48447 files belonging to 10 classes.
Using 41180 files for training.
Found 48447 files belonging to 10 classes.
Using 7267 files for validation.
Epoch 1/15
1287/1287 ━━━━━━━━━━━━━━━━━━━━ 0s 886ms/step - accuracy: 0.7406 - loss: 0.7371 - top3_accuracy: 0.9510
Epoch 1: val_accuracy improved from None to 0.74006, saving model to /content/drive/MyDrive/art_classification/checkpoints/ResNet50_best.keras

Epoch 1: finished saving model to /content/drive/MyDrive/art_classification/checkpoints/ResNet50_best.keras
1287/1287 ━━━━━━━━━━━━━━━━━━━━ 1302s 980ms/step - accuracy: 0.7523 - loss: 0.7155 - top3_accuracy: 0.9541 - val_accuracy: 0.7401 - val_loss: 0.7552 - val_top3_accuracy: 0.9401 - learning_rate: 1.0000e-05
Epoch 2/15
1287/1287 ━━━━━━━━━━━━━━━━━━━━ 0s 874ms/step - accuracy: 0.7610 - loss: 0.6800 - 

# 5. Сравнение моделей — визуализация обучения

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from pathlib import Path

plt.rcParams["figure.dpi"] = 120
sns.set_theme(style="whitegrid", palette="mako")

CKPT_DIR = Path("/content/drive/MyDrive/art_classification/checkpoints")
MODELS   = ["EfficientNetB0", "ResNet50", "Xception"]
COLORS   = {"EfficientNetB0": "#2196F3", "ResNet50": "#FF9800", "Xception": "#4CAF50"}

# 1. Загрузка CSV-историй
histories = {}

for name in MODELS:
    csv_path = CKPT_DIR / f"{name}_history.csv"
    if not csv_path.exists():
        print(f"  Файл не найден: {csv_path}")
        continue
    df = pd.read_csv(csv_path)
    histories[name] = df
    print(f"✓ {name}: {len(df)} эпох загружено")

if not histories:
    raise FileNotFoundError("Не найдено ни одного файла истории! Дождитесь окончания обучения.")

# 2. Определение границы фаз
def detect_phase_boundary(df, default=7):
    """Находит эпоху начала fine-tuning по падению lr."""
    if "lr" not in df.columns:
        return default
    lr_values = df["lr"].values
    for i in range(1, len(lr_values)):
        if lr_values[i - 1] / max(lr_values[i], 1e-12) > 5:
            return i
    return default

phase_boundaries = {}
for name, df in histories.items():
    boundary = detect_phase_boundary(df)
    phase_boundaries[name] = boundary
    print(f"  {name}: фаза 2 начинается с эпохи {boundary + 1}")

In [ ]:
# Построение графиков
plot_config = [
    ("loss",          "Потери — Train",       "Loss"),
    ("val_loss",      "Потери — Validation",  "Loss"),
    ("accuracy",      "Точность — Train",     "Accuracy"),
    ("val_accuracy",  "Точность — Validation","Accuracy"),
]

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle("Сравнение обучения трёх архитектур", fontsize=16, y=1.02)

for ax, (col, title, ylabel) in zip(axes.flat, plot_config):
    for name, df in histories.items():
        epochs = range(1, len(df) + 1)
        ax.plot(epochs, df[col], label=name,
                color=COLORS[name], linewidth=1.8)

        # Вертикальная линия — граница фаз
        boundary = phase_boundaries[name]
        ax.axvline(x=boundary + 0.5, color=COLORS[name],
                   linestyle=":", linewidth=1.0, alpha=0.6)

    ax.set_title(title, fontsize=13)
    ax.set_xlabel("Эпоха", fontsize=11)
    ax.set_ylabel(ylabel, fontsize=11)
    ax.legend(fontsize=9)
    ax.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))
    ax.grid(True, alpha=0.3)

# Общая аннотация для пунктира
fig.text(0.5, -0.01,
         "Пунктир — начало fine-tuning (разморозка backbone)",
         ha="center", fontsize=10, style="italic", color="gray")

plt.tight_layout()
plt.savefig(CKPT_DIR / "training_curves.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"\nГрафик сохранён: {CKPT_DIR / 'training_curves.png'}")

In [ ]:
# Сводная таблица
print("\n" + "=" * 65)
print(f"{'Модель':<20} {'Val Accuracy':>14} {'Эпоха':>8} {'Val Loss':>10}")
print("-" * 65)

best_model_name = None
best_val_acc    = 0.0

for name, df in histories.items():
    best_idx   = df["val_accuracy"].idxmax()
    best_acc   = df.loc[best_idx, "val_accuracy"]
    best_loss  = df.loc[best_idx, "val_loss"]
    best_epoch = best_idx + 1  # CSV индексация с 0

    if best_acc > best_val_acc:
        best_val_acc    = best_acc
        best_model_name = name

    print(f"{name:<20} {best_acc:>13.4f} {best_epoch:>8} {best_loss:>10.4f}")

print("-" * 65)
print(f"\n Лучшая модель: {best_model_name} (val_accuracy = {best_val_acc:.4f})")
print("=" * 65)

# 6. Оценка на тестовой выборке + матрица ошибок

In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

from sklearn.metrics import (
    accuracy_score,
    top_k_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
)

# Настройки
plt.rcParams["figure.dpi"] = 120
sns.set_theme(style="whitegrid", palette="mako")

CKPT_DIR    = Path("/content/drive/MyDrive/art_classification/checkpoints")
DATASET_DIR = Path("/content/dataset_top10")
TEST_DIR    = DATASET_DIR / "test"
BATCH_SIZE  = 32

BACKBONE_CONFIG = {
    "ResNet50": {
        "preprocess_fn": tf.keras.applications.resnet50.preprocess_input,
        "img_size":      (224, 224),
    },
    "Xception": {
        "preprocess_fn": tf.keras.applications.xception.preprocess_input,
        "img_size":      (299, 299),
    },
    "EfficientNetB0": {
        "preprocess_fn": tf.keras.applications.efficientnet.preprocess_input,
        "img_size":      (224, 224),
    },
}

MODELS = ["EfficientNetB0", "ResNet50", "Xception"]

def build_test_ds(preprocess_fn, img_size):
    """Создаёт test_ds с правильной предобработкой."""
    raw_ds = tf.keras.utils.image_dataset_from_directory(
        TEST_DIR,
        image_size=img_size,
        batch_size=BATCH_SIZE,
        shuffle=False, # КРИТИЧЕСКИ ВАЖНО: отключено перемешивание!
        label_mode="categorical",
    )
    class_names = raw_ds.class_names
    test_ds = (
        raw_ds
        .map(lambda x, y: (preprocess_fn(x), y),
             num_parallel_calls=tf.data.AUTOTUNE)
        .prefetch(tf.data.AUTOTUNE)
    )
    return test_ds, class_names

def get_predictions(model, test_ds):
    """Получает предсказания и истинные метки из test_ds."""
    y_prob_list = []
    y_true_list = []
    for images, labels in test_ds:
        probs = model.predict(images, verbose=0)
        y_prob_list.append(probs)
        y_true_list.append(labels.numpy())

    y_prob = np.concatenate(y_prob_list, axis=0)
    y_true_onehot = np.concatenate(y_true_list, axis=0)
    y_true = np.argmax(y_true_onehot, axis=1)
    y_pred = np.argmax(y_prob, axis=1)
    return y_true, y_pred, y_prob

print("Функции для оценки успешно загружены.")

**Оценка каждой модели**

In [ ]:
results = []            # для сводной таблицы
cm_data = {}            # confusion matrices для графиков
class_names = None      # будет заполнено при первом вызове

for name in MODELS:
    ckpt_path = CKPT_DIR / f"{name}_best.keras"
    if not ckpt_path.exists():
        print(f" Чекпоинт не найден: {ckpt_path}")
        continue

    print(f"\n{'=' * 60}\n  Оценка: {name}\n{'=' * 60}")
    cfg = BACKBONE_CONFIG[name]

    model = keras.models.load_model(str(ckpt_path))
    print(f"  Модель загружена: {ckpt_path.name}")

    test_ds, class_names = build_test_ds(cfg["preprocess_fn"], cfg["img_size"])


    y_true, y_pred, y_prob = get_predictions(model, test_ds)
    print(f"  Тестовых изображений: {len(y_true)}")

    # Метрики
    acc      = accuracy_score(y_true, y_pred)
    top3_acc = top_k_accuracy_score(y_true, y_prob, k=3)
    prec     = precision_score(y_true, y_pred, average="macro", zero_division=0)
    rec      = recall_score(y_true, y_pred, average="macro", zero_division=0)
    f1       = f1_score(y_true, y_pred, average="macro", zero_division=0)

    results.append({
        "Модель":         name,
        "Accuracy":       acc,
        "Top-3 Accuracy": top3_acc,
        "Precision":      prec,
        "Recall":         rec,
        "F1-score":       f1,
    })

    print(f"\n  Accuracy:       {acc:.4f}")
    print(f"  Top-3 Accuracy: {top3_acc:.4f}")
    print(f"  Precision:      {prec:.4f}")
    print(f"  Recall:         {rec:.4f}")
    print(f"  F1-score:       {f1:.4f}")

    print(f"\n  Classification Report ({name}):")
    print(classification_report(y_true, y_pred, target_names=class_names, digits=3))

    # сonfusion matrix
    cm = confusion_matrix(y_true, y_pred, normalize="true")
    cm_data[name] = cm

    del model
    tf.keras.backend.clear_session()

print("\nСбор метрик завершен! Данные готовы к визуализации.")

In [ ]:
# ЯЧЕЙКА 11: Графики Confusion Matrix (1×3)
if cm_data:
    n_models = len(cm_data)
    fig, axes = plt.subplots(1, n_models, figsize=(7.5 * n_models, 7))
    if n_models == 1:
        axes = [axes]

    fig.suptitle("Матрицы ошибок на тестовой выборке (нормализованные)",
                 fontsize=15, y=1.03)

    for ax, (name, cm) in zip(axes, cm_data.items()):
        sns.heatmap(
            cm, annot=True, fmt=".2f", cmap="mako",
            xticklabels=class_names,
            yticklabels=class_names,
            vmin=0, vmax=1,
            linewidths=0.5, linecolor="white",
            cbar_kws={"shrink": 0.8},
            ax=ax,
        )
        ax.set_title(name, fontsize=13, pad=10)
        ax.set_xlabel("Предсказанный стиль", fontsize=10)
        ax.set_ylabel("Истинный стиль", fontsize=10)
        ax.tick_params(axis="both", labelsize=8)
        ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha="right")
        ax.set_yticklabels(ax.get_yticklabels(), rotation=0)

    plt.tight_layout()
    save_path = CKPT_DIR / "confusion_matrices.png"
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"\nГрафики сохранены: {save_path}")
else:
    print("Нет данных для отрисовки матриц ошибок.")

In [ ]:
# Сводная таблица
if results:
    df_results = pd.DataFrame(results)

    print("\n" + "=" * 75)
    print("СВОДНАЯ ТАБЛИЦА — ТЕСТОВАЯ ВЫБОРКА")
    print("=" * 75)

    df_display = df_results.copy()
    for col in ["Accuracy", "Top-3 Accuracy", "Precision", "Recall", "F1-score"]:
        df_display[col] = df_display[col].map(lambda x: f"{x:.4f}")

    print(df_display.to_string(index=False))
    print("=" * 75)

    best_idx  = df_results["F1-score"].idxmax()
    best_row  = df_results.loc[best_idx]

    print(f"\n🏆 Рекомендация: {best_row['Модель']}")
    print(f"   F1-score = {best_row['F1-score']:.4f}  |  "
          f"Accuracy = {best_row['Accuracy']:.4f}  |  "
          f"Top-3 = {best_row['Top-3 Accuracy']:.4f}")
    print(f"\n   F1-score выбран как основная метрика, поскольку он учитывает")
    print(f"   и полноту, и точность — это важно при несбалансированных классах.")

    csv_path = CKPT_DIR / "test_results_comparison.csv"
    df_results.to_csv(csv_path, index=False)
    print(f"\n   Таблица сохранена: {csv_path}")

# 7. Анализ ошибок — самые сложные пары классов

In [ ]:
# Поиск самых путаемых пар стилей
import numpy as np
import tensorflow as tf
from tensorflow import keras
from pathlib import Path
from PIL import Image

import matplotlib.pyplot as plt
import seaborn as sns

plt.rcParams["figure.dpi"] = 120
sns.set_theme(style="whitegrid")

CKPT_DIR    = Path("/content/drive/MyDrive/art_classification/checkpoints")
TEST_DIR    = Path("/content/dataset_top10/test")
BATCH_SIZE  = 32

# Восстановление после перезапуска ядра
if "class_names" not in dir() or class_names is None:
    class_names = sorted([d.name for d in TEST_DIR.iterdir() if d.is_dir()])
    print(f"class_names восстановлены из {TEST_DIR}: {class_names}")

# 1. Определяем лучшую модель
try:
    best_idx   = df_results["F1-score"].idxmax()
    BEST_MODEL = df_results.loc[best_idx, "Модель"]
    print(f"Лучшая модель (из шага 6): {BEST_MODEL}")
except NameError:
    BEST_MODEL = "EfficientNetB0"   # ← на случай перезапуска сессии
    print(f"Внимание: df_results не найден. Используем модель по умолчанию: {BEST_MODEL}")

cfg      = BACKBONE_CONFIG[BEST_MODEL]
img_size = cfg["img_size"]

if "cm_data" not in dir():
    raise RuntimeError("cm_data не найден. Сначала выполните шаг 6: оценка на тестовой выборке.")

# 2. Находим топ-5 самых путаемых пар (off-diagonal)
cm = cm_data[BEST_MODEL]

# Зануляем диагональ — нас интересуют только ошибки
cm_offdiag = cm.copy()
np.fill_diagonal(cm_offdiag, 0)

top_k = 5
flat_indices = np.argsort(cm_offdiag.ravel())[::-1][:top_k]
pairs = []

print(f"\nТоп-{top_k} самых путаемых пар ({BEST_MODEL}):")
print("-" * 65)
print(f"{'Истинный стиль':<28} → {'Предсказанный':<28} Доля")
print("-" * 65)

for flat_idx in flat_indices:
    true_idx = flat_idx // cm.shape[1]
    pred_idx = flat_idx %  cm.shape[1]
    value    = cm_offdiag[true_idx, pred_idx]
    pairs.append((true_idx, pred_idx, value))
    print(f"{class_names[true_idx]:<28} → {class_names[pred_idx]:<28} {value:.3f}")

print("-" * 65)

In [ ]:
# Сбор ошибочных изображений
print(f"Загрузка модели {BEST_MODEL} для анализа ошибок...")
model = keras.models.load_model(str(CKPT_DIR / f"{BEST_MODEL}_best.keras"))
preprocess_fn = cfg["preprocess_fn"]

# Собираем ВСЕ тестовые пути + метки
test_ds_raw = tf.keras.utils.image_dataset_from_directory(
    TEST_DIR,
    image_size=img_size,
    batch_size=BATCH_SIZE,
    shuffle=False,
    label_mode="categorical",
)
ds_class_names = test_ds_raw.class_names
file_paths = test_ds_raw.file_paths

# Предобработка
test_ds = (
    test_ds_raw
    .map(lambda x, y: (preprocess_fn(x), y), num_parallel_calls=tf.data.AUTOTUNE)
    .prefetch(tf.data.AUTOTUNE)
)

y_prob_list, y_true_list = [], []
for images, labels in test_ds:
    probs = model.predict(images, verbose=0)
    y_prob_list.append(probs)
    y_true_list.append(labels.numpy())

y_prob = np.concatenate(y_prob_list, axis=0)
y_true = np.argmax(np.concatenate(y_true_list, axis=0), axis=1)
y_pred = np.argmax(y_prob, axis=1)

print(f"\nВсего тестовых изображений: {len(y_true)}")
print(f"Всего ошибок: {np.sum(y_true != y_pred)}")

EXAMPLES_PER_PAIR = 3
error_examples = {}

for true_idx, pred_idx, _ in pairs:
    mask = (y_true == true_idx) & (y_pred == pred_idx)
    error_indices = np.where(mask)[0]

    examples = []
    # Берём примеры с наивысшей уверенностью в неправильном ответе
    confidences = y_prob[error_indices, pred_idx]
    sorted_order = np.argsort(confidences)[::-1]

    for rank in sorted_order[:EXAMPLES_PER_PAIR]:
        idx = error_indices[rank]
        examples.append({
            "file_path":  file_paths[idx],
            "true_label": ds_class_names[true_idx],
            "pred_label": ds_class_names[pred_idx],
            "confidence": y_prob[idx, pred_idx],
            "true_conf":  y_prob[idx, true_idx],
        })

    error_examples[(true_idx, pred_idx)] = examples
    print(f"  {ds_class_names[true_idx]} → {ds_class_names[pred_idx]}: "
          f"найдено {len(error_indices)} ошибок, собрано {len(examples)} примеров")

In [ ]:
# Визуализация ошибок
n_pairs   = len(pairs)
n_cols    = EXAMPLES_PER_PAIR
fig, axes = plt.subplots(n_pairs, n_cols, figsize=(5 * n_cols, 4.5 * n_pairs))

fig.suptitle(
    f"Анализ ошибок: топ-{top_k} путаемых пар ({BEST_MODEL})",
    fontsize=16, y=1.01, fontweight="bold"
)

for row, (true_idx, pred_idx, cm_val) in enumerate(pairs):
    examples = error_examples[(true_idx, pred_idx)]

    for col in range(n_cols):
        ax = axes[row, col] if n_pairs > 1 else axes[col]

        if col < len(examples):
            ex = examples[col]
            img = Image.open(ex["file_path"]).convert("RGB")
            ax.imshow(img)
            ax.set_title(
                f"Предсказано: {ex['pred_label']}\nУверенность: {ex['confidence']:.1%}",
                fontsize=10, color="darkred", fontweight="bold",
            )
        else:
            ax.text(0.5, 0.5, "Нет\nпримера",
                    ha="center", va="center", fontsize=10, color="gray",
                    transform=ax.transAxes)

        # ИСПРАВЛЕНИЕ: Прячем насечки, но оставляем возможность писать ylabel
        ax.set_xticks([])
        ax.set_yticks([])

    # Подпись строки слева
    row_ax = axes[row, 0] if n_pairs > 1 else axes[0]
    row_ax.set_ylabel(
        f"Истинный класс:\n{ds_class_names[true_idx]}\n({cm_val:.1%} ошибок)",
        fontsize=11, rotation=0, labelpad=70, va="center", ha="right",
        fontweight="bold",
    )

plt.tight_layout()
save_path = CKPT_DIR / "error_analysis_top5.png"
plt.savefig(save_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"\nГрафик сохранён: {save_path}")

 # 8. Экспорт лучшей модели

In [ ]:
import json
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from pathlib import Path

CKPT_DIR   = Path("/content/drive/MyDrive/art_classification/checkpoints")
EXPORT_DIR = Path("/content/drive/MyDrive/art_classification/production_model")
TEST_DIR   = Path("/content/dataset_top10/test")
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

PREPROCESS_PATHS = {
    "ResNet50":       "tf.keras.applications.resnet50.preprocess_input",
    "Xception":       "tf.keras.applications.xception.preprocess_input",
    "EfficientNetB0": "tf.keras.applications.efficientnet.preprocess_input",
}

if "class_names" not in dir() or class_names is None:
    class_names = sorted([d.name for d in TEST_DIR.iterdir() if d.is_dir()])
    print(f"class_names восстановлены из {TEST_DIR}: {class_names}")

if "df_results" not in dir():
    csv_path = CKPT_DIR / "test_results_comparison.csv"
    if csv_path.exists():
        df_results = pd.read_csv(csv_path)
        print(f"df_results восстановлен из {csv_path}")
    else:
        print("Запустите сначала шаг 6")

# 1. Определяем лучшую модель
try:
    best_idx   = df_results["F1-score"].idxmax()
    BEST_MODEL = df_results.loc[best_idx, "Модель"]
    best_acc   = df_results.loc[best_idx, "Accuracy"]
    best_f1    = df_results.loc[best_idx, "F1-score"]
    print(f"Лучшая модель (из шага 6): {BEST_MODEL}")
except NameError:
    BEST_MODEL = "EfficientNetB0"     # ← замените, если нужно
    best_acc, best_f1 = None, None
    print(f"Используем модель по умолчанию: {BEST_MODEL}")

cfg      = BACKBONE_CONFIG[BEST_MODEL]
img_size = cfg["img_size"]

# Загрузка чекпоинта
ckpt_path = CKPT_DIR / f"{BEST_MODEL}_best.keras"
model = keras.models.load_model(str(ckpt_path))
print(f"Модель загружена: {ckpt_path.name}")

# --- Вспомогательная функция для размера файлов ---
def get_size_mb(path):
    p = Path(path)
    if p.is_file(): return p.stat().st_size / (1024 * 1024)
    elif p.is_dir(): return sum(f.stat().st_size for f in p.rglob("*") if f.is_file()) / (1024 * 1024)
    return 0.0

# 2. Формат .keras (Keras v3)
keras_path = EXPORT_DIR / f"{BEST_MODEL}_production.keras"
model.save(str(keras_path))

# 3. Формат TensorFlow SavedModel
saved_model_path = EXPORT_DIR / f"{BEST_MODEL}_saved_model"
tf.saved_model.save(model, str(saved_model_path))

# 4. Формат TFLite (Опционально)
tflite_path = EXPORT_DIR / f"{BEST_MODEL}_model.tflite"
try:
    converter = tf.lite.TFLiteConverter.from_keras_model(model)
    converter.optimizations = [tf.lite.Optimize.DEFAULT]
    with open(tflite_path, "wb") as f:
        f.write(converter.convert())
except Exception as e:
    print(f" TFLite конвертация не удалась: {e}")
    tflite_path = None

# 5. JSON с метаинформацией
val_accuracy = None
csv_path = CKPT_DIR / f"{BEST_MODEL}_history.csv"
if csv_path.exists():
    hist_df = pd.read_csv(csv_path)
    val_accuracy = float(hist_df["val_accuracy"].max())

meta = {
    "backbone_name":  BEST_MODEL,
    "class_names":    class_names,
    "num_classes":    len(class_names),
    "img_size":       list(img_size),
    "input_shape":    list(img_size) + [3],
    "val_accuracy":   val_accuracy,
    "test_accuracy":  float(best_acc) if best_acc is not None else None,
    "test_f1_score":  float(best_f1)  if best_f1  is not None else None,
    "formats": {
        "keras":       str(keras_path.name),
        "saved_model": str(saved_model_path.name),
        "tflite":      str(tflite_path.name) if tflite_path else None,
    },
    "preprocessing":  PREPROCESS_PATHS[BEST_MODEL],
}

meta_path = EXPORT_DIR / "model_meta.json"
with open(meta_path, "w", encoding="utf-8") as f:
    json.dump(meta, f, ensure_ascii=False, indent=2)

# 6. Вывод итогов
print(f"\n{'=' * 50}\n{'Формат':<20} {'Размер':>10}\n{'-' * 50}")
print(f"{'Keras (.keras)':<20} {get_size_mb(keras_path):>8.1f} МБ")
print(f"{'SavedModel (папка)':<20} {get_size_mb(saved_model_path):>8.1f} МБ")
if tflite_path and tflite_path.exists():
    print(f"{'TFLite (.tflite)':<20} {get_size_mb(tflite_path):>8.1f} МБ")
print(f"{'Метаданные (.json)':<20} {get_size_mb(meta_path):>8.1f} МБ\n{'=' * 50}")
print(f"\n✓ Всё сохранено в: {EXPORT_DIR}")

In [ ]:
# Функция инференса

from PIL import Image
import matplotlib.pyplot as plt

def predict_single_image(image_path, demo_model, preprocess_fn, class_names, img_size=(224, 224), top_k=3):
    # Загрузка и предобработка
    img = Image.open(image_path).convert("RGB")
    img_resized = img.resize((img_size[1], img_size[0]))
    img_array = np.array(img_resized, dtype=np.float32)
    img_array = np.expand_dims(img_array, axis=0)
    img_array = preprocess_fn(img_array)

    # Предсказание
    probs = demo_model.predict(img_array, verbose=0)[0]

    # Топ-K
    top_indices = np.argsort(probs)[::-1][:top_k]
    predictions = [(class_names[i], float(probs[i])) for i in top_indices]

    # Визуализация
    fig, (ax_img, ax_bar) = plt.subplots(1, 2, figsize=(12, 4.5), gridspec_kw={"width_ratios": [1, 1.2]})

    ax_img.imshow(img)
    ax_img.axis("off")
    ax_img.set_title(Path(image_path).name, fontsize=11)

    names  = [p[0] for p in predictions][::-1]
    values = [p[1] for p in predictions][::-1]
    colors = ["#2196F3"] * len(names)
    colors[-1] = "#4CAF50"  # лучший — зелёный

    bars = ax_bar.barh(names, values, color=colors, edgecolor="white", height=0.6)
    for bar, val in zip(bars, values):
        ax_bar.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height() / 2,
                    f"{val:.1%}", va="center", fontsize=10)

    ax_bar.set_xlim(0, 1.0)
    ax_bar.set_xlabel("Вероятность", fontsize=11)
    ax_bar.set_title(f"Топ-{top_k} предсказаний", fontsize=12)
    ax_bar.grid(axis="x", alpha=0.3)

    plt.tight_layout()
    plt.show()

    return predictions

print("Функция predict_single_image готова к работе!")

In [ ]:
# Запуск демо
import random

print("\n" + "=" * 60)
print("ДЕМО: предсказание на случайных тестовых изображениях")
print("=" * 60)

TEST_DIR = Path("/content/dataset_top10/test")
test_images = list(TEST_DIR.rglob("*.jpg"))
if not test_images:
    test_images = list(TEST_DIR.rglob("*.png"))

random.seed(42)
demo_samples = random.sample(test_images, min(3, len(test_images)))
preprocess_fn = cfg["preprocess_fn"]

for img_path in demo_samples:
    true_style = img_path.parent.name
    print(f"\n{'─' * 60}")
    print(f"Истинный стиль: {true_style}")
    preds = predict_single_image(
        image_path=img_path,
        demo_model=model,
        preprocess_fn=preprocess_fn,
        class_names=class_names,
        img_size=img_size,
        top_k=3,
    )

# Очистка видеопамяти после всех тестов
del model
tf.keras.backend.clear_session()
print(f"\n✓ Очистка завершена. Проект готов к деплою!")